# Group By

Grouping is a fundamental concept when you need to perform operations
**separately** for different subsets of your data. In tidypolars-extra,
grouping is done via the `by` parameter available on `filter`, `mutate`,
and `summarize`.

For example, in the `mtcars` dataset there are 3 possible values for
cylinders (`cyl`). You can group by `cyl` to perform operations separately
for each of these 3 groups of rows.

In [ ]:
import os, pathlib
_DATA_DIR = str(pathlib.Path(__import__('tidypolars_extra').__file__).parent / 'data')

import tidypolars_extra as tp

mtcars = tp.tibble(tp.read_data(fn=f"{_DATA_DIR}/mtcars.csv", sep=",", silently=True))

small_cars = mtcars.select("name", "cyl", "gear", "hp")

small_cars

## Grouping with `by`

The `by` parameter is supported directly on `filter`, `mutate`, and
`summarize`. This is often more concise than a separate `group_by` step.

### Grouped filter

Keep rows where `hp` is greater than the mean `hp` **within each `cyl` group**:

In [ ]:
small_cars.filter(tp.col("hp") > tp.col("hp").mean(), by="cyl")

### Grouped mutate

Compute the average `hp` within each `cyl` group and attach it to every row:

In [ ]:
small_cars.mutate(avg_hp=tp.col("hp").mean(), by="cyl")

### Grouped summarize

Compute summary statistics per group:

In [ ]:
small_cars.summarize(avg_hp=tp.col("hp").mean(), by="cyl")

## Grouping by multiple columns

Pass a list of column names to `by` to group by multiple columns:

In [ ]:
small_cars.summarize(avg_hp=tp.col("hp").mean(), by=["cyl", "gear"])

## Using `group_by` explicitly

You can also use the `.group_by()` method, which returns a `TibbleGroupBy`
object. This object supports `filter`, `mutate`, and `summarize`:

In [ ]:
g_cyl = mtcars.group_by("cyl")

# Summarize per group
g_cyl.summarize(avg_hp=tp.col("hp").mean())

> **Note:** The `by` parameter on `filter`, `mutate`, and `summarize` is typically
preferred for its conciseness and clarity. Both approaches produce the
same result.

## Practical example: top N per group

A common pattern is to find the "top N" rows within each group. For example,
the 2 cars with the lowest `hp` in each `cyl` group:

In [ ]:
(
    small_cars
    .arrange("hp")
    .mutate(row_num=tp.row_number(), by="cyl")
    .filter(tp.col("row_num") <= 2)
    .drop("row_num")
)